In [ ]:
import os, sys
from IPython.display import Markdown, display
repo_path = os.path.abspath(os.path.join(os.getcwd(), '../../..'))
code_path = os.path.abspath(os.path.join(repo_path, 'code/src'))
os.chdir(repo_path)
if code_path not in sys.path:
    sys.path.append(code_path)
display(Markdown(f'**Repository root:** {repo_path}'))

# ODD Protocol for the Pneumococcal ABM


## Table of Contents

1. [Overview](#overview)
2. [Design Concepts](#design-concepts)
3. [Details](#details)
4. [Code-Verified Single Timestep Trace](#single-timestep-trace)


# 1. Overview <a name="overview"></a>

## 1.1 Purpose

The model is an individual-level, age-structured, multi-serotype transmission model with vaccination and disease outcomes.
The class hierarchy is `Disease` → `VaryingTransmissionDisease` → `DiseaseModel`,
combining demography with serotype-group-specific transmission:
[model/disease/disease.py](../model/disease/disease.py) (line 21),
[model/disease/varying_transmission_disease.py](../model/disease/varying_transmission_disease.py) (line 25),
[model/disease/disease_simulation.py](../model/disease/disease_simulation.py) (line 115).

## 1.2 Entities, state variables, and scales

### Entities
- Individual agents are rows of `P.I` (a Polars DataFrame) in `DisPopulation`:
  [model/population/disease_population.py](../model/population/disease_population.py) (line 15).
- The disease process is managed by `VaryingTransmissionDisease` and wrapped by `DiseaseModel` which adds observers:
  [model/disease/varying_transmission_disease.py](../model/disease/varying_transmission_disease.py) (line 25),
  [run_scenarios/varying_disease_model.py](../run_scenarios/varying_disease_model.py) (line 41).

### Agent-level state variables (per individual)

**Demography**
- `id`: unique integer identifier.
- `age`: age in whole years.
- `age_days`: days elapsed since last birthday (0–363); together with `age` this gives sub-year precision.
- `age_group`: integer 0–14 (= `age // 5`), capped at 15 for individuals aged 80+;
  used to index the contact matrix and FOI calculation:
  [model/population/disease_population.py](../model/population/disease_population.py) (line 127).

**Per-individual random draws** (three distinct columns)
- `quantile`: uniform [0, 1] drawn once per individual at creation and **never refreshed**;
  maps each agent to a fixed antibody-level percentile in the lognormal distribution,
  representing persistent between-person heterogeneity in vaccine response:
  [model/population/disease_population.py](../model/population/disease_population.py) (line 126),
  [model/disease/disease.py](../model/disease/disease.py) (lines 1027, 1214).
- `random`: uniform [0, 1] that is **refreshed after every use** (death check, exposure comparison,
  disease-outcome draw). It acts as a single-use random ticket consumed at each stochastic step:
  [model/disease/disease_simulation.py](../model/disease/disease_simulation.py) (line 253),
  [model/disease/disease.py](../model/disease/disease.py) (lines 957, 960, 966, 1290).
- `exp_random`: exponential random over the age-specific mean carriage duration, **re-sampled
  at each acquisition event** by `generate_duration_of_infection()`;
  converted to discrete clearance day as `day + period * (exp_random / period).round()`
  and appended to `endTimes`:
  [model/disease/disease.py](../model/disease/disease.py) (lines 1404, 1100).

**Vaccination** (stored as a Polars struct column `vaccines`)
- Fields: `no_of_doses`, `on_time`, `vaccine_type`, `final_vaccine_time`:
  [model/population/disease_population.py](../model/population/disease_population.py) (line 128).

**Infection history / carriage**
- `no_of_strains`: number of strains currently carried (0 = susceptible).
- `strain_list`: list of carried strain IDs.
- `endTimes`: list of clearance days (one per carried strain).
- `no_past_infections`: cumulative count of all past acquisitions:
  [model/population/disease_population.py](../model/population/disease_population.py) (lines 131–141).

### Time scale
- The model runs in discrete ticks; one tick is `364 // t_per_year` days:
  [model/disease/disease_simulation.py](../model/disease/disease_simulation.py) (line 226).
- Baseline parameters define weekly time steps with `t_per_year = 52`:
  [run_scenarios/base_params.py](../run_scenarios/base_params.py).
- Day-of-year arithmetic uses 364 (not 365), so a year is exactly 52 × 7 days.

### Space / contact structure
- Interaction is age-mixing through a contact matrix loaded at runtime from
  `data/population/all_contact_matrix_Australia_prem_2017.csv` and wrapped in `KnownContactMatrix`:
  [run_scenarios/varying_disease_model.py](../run_scenarios/varying_disease_model.py) (line 74),
  [model/disease/contact_matrix.py](../model/disease/contact_matrix.py).

## 1.3 Process overview and scheduling

At each simulation tick in `DisSimulation._main_loop`, the order is:

1. Compute `day = t * 364 // t_per_year`:
   [model/disease/disease_simulation.py](../model/disease/disease_simulation.py) (line 133).
2. Update demography (ageing / death via `random` vs death-rate lookup / births / migration):
   [model/disease/disease_simulation.py](../model/disease/disease_simulation.py) (line 137).
3. Call `Disease.update()` (which delegates FOI to `VaryingTransmissionDisease.calc_foi`):
   [model/disease/disease_simulation.py](../model/disease/disease_simulation.py) (line 143).

Inside `Disease.update()`, the disease-related sequence is:

1. Vaccination rollout checks (`check_vaccines`):
   [model/disease/disease.py](../model/disease/disease.py) (lines 515, 534).
2. Community FOI computation (`VaryingTransmissionDisease.calc_foi`) + exposure sampling (`check_exposure`):
   [model/disease/disease.py](../model/disease/disease.py) (lines 521, 927).
3. External introductions (`external_exposure`):
   [model/disease/disease.py](../model/disease/disease.py) (lines 525, 1354).
4. Individual state update for newly infected (`update_ind_states`):
   [model/disease/disease.py](../model/disease/disease.py) (lines 528, 1468).
5. Observer recording:
   [model/disease/disease.py](../model/disease/disease.py) (line 1478).


# 2. Design Concepts <a name="design-concepts"></a>

## 2.1 Basic principles

- Transmission is age-structured and **serotype-group-specific**.
  Serotypes are classified into four vaccine groups — pcv7, pcv13, ppv23 (PPV23 serotypes not in PCV7/13),
  and nonppv23 — each with its own transmission multiplier from `base_params.py`:
  [model/disease/varying_transmission_disease.py](../model/disease/varying_transmission_disease.py) (lines 52, 54, 86).
- For each group, FOI is built from age-group infection fractions and the contact matrix,
  then transformed as $1-e^{-FOI}$:
  [model/disease/varying_transmission_disease.py](../model/disease/varying_transmission_disease.py) (lines 376, 411).
- Co-infection is constrained by `max_no_coinfections`; susceptibility is reduced for already-infected individuals:
  [model/disease/disease.py](../model/disease/disease.py) (line 957).
- Antibody waning and protection are computed via the `antibody_levels` module:
  [model/disease/disease.py](../model/disease/disease.py) (line 20),
  [model/disease/antibody_levels.py](../model/disease/antibody_levels.py).

## 2.2 Emergence

- Serotype prevalence emerges from group-specific FOI-driven transmission plus external seeding
  and co-infection constraints (`no_of_strains` limit):
  [model/disease/varying_transmission_disease.py](../model/disease/varying_transmission_disease.py) (lines 448, 472).
- IPD/CAP incidence by age and vaccine strata is an emergent output recorded by observers:
  [model/disease/disease.py](../model/disease/disease.py) (line 1141),
  [model/observers/obs_disease_by_age.py](../model/observers/obs_disease_by_age.py).

## 2.3 Adaptation and objectives

- Agents do not explicitly optimize behavior. Their state changes are rule-driven
  (vaccination eligibility / schedule, infection / recovery, ageing, death, migration):
  [model/disease/disease.py](../model/disease/disease.py) (lines 534, 927, 1468),
  [model/disease/disease_simulation.py](../model/disease/disease_simulation.py) (line 212).

## 2.4 Learning and prediction

- No explicit learning or memory update policy: cumulative exposure (`no_past_infections`) and
  vaccine history fields encode past events but do not drive adaptive decisions:
  [model/population/disease_population.py](../model/population/disease_population.py) (line 141).
- Vaccine impact is modeled through antibody-mediated probabilistic protection
  via logistic/lognormal functions, not adaptive policies:
  [model/disease/disease.py](../model/disease/disease.py) (lines 101, 127, 210).

## 2.5 Sensing and interaction

- Individuals do not directly sense neighbors; interaction is mediated through age-contact mixing
  and **group-specific** prevalence (pcv7/pcv13/ppv23/nonppv23) in the FOI calculation:
  [model/disease/varying_transmission_disease.py](../model/disease/varying_transmission_disease.py) (lines 411, 448).
- Vaccination interaction with infection occurs via the `vaccines` struct and the
  `vaccine_antibody_df` lookup table joined at exposure and disease-outcome steps:
  [model/disease/disease.py](../model/disease/disease.py) (lines 999, 1150).

## 2.6 Stochasticity

Three separate random columns serve distinct roles:

- **`quantile`** (persistent): drawn once at birth/migration, never refreshed.
  Maps each individual to a fixed percentile in the antibody lognormal distribution,
  reflecting stable between-person heterogeneity in vaccine immunogenicity:
  [model/population/disease_population.py](../model/population/disease_population.py) (line 126),
  [model/disease/disease.py](../model/disease/disease.py) (lines 1027, 1214).
- **`random`** (ephemeral, refreshed after every use): single-use uniform ticket;
  consumed sequentially for death check (vs age-specific death rate), exposure probability
  comparison, and disease-outcome draw, then immediately replaced:
  [model/disease/disease_simulation.py](../model/disease/disease_simulation.py) (lines 253, 256),
  [model/disease/disease.py](../model/disease/disease.py) (lines 950, 957, 960, 966, 1290).
- **`exp_random`** (refreshed per acquisition): exponential variate over the age-specific mean
  carriage duration, re-drawn at each strain acquisition by `generate_duration_of_infection()`;
  used to set the discrete clearance day appended to `endTimes`:
  [model/disease/disease.py](../model/disease/disease.py) (lines 1404, 1100).

Additional randomness sources: RNG streams for seeding, vaccination timing, strain assignment,
and external exposure:
[model/disease/disease.py](../model/disease/disease.py) (lines 286, 318, 1354, 1404).

## 2.7 Observation

`DiseaseModel` attaches observers for population, prevalence, vaccination rollout scenarios,
disease by age, vaccines delivered, and prevalence by age:
[run_scenarios/varying_disease_model.py](../run_scenarios/varying_disease_model.py) (lines 50, 62).

Examples of recorded outputs:

- Population size and age distribution:
  [model/observers/obs_pop.py](../model/observers/obs_pop.py).
- Overall prevalence and serotype fractions:
  [model/observers/obs_prevalence.py](../model/observers/obs_prevalence.py).
- Vaccination rollout scenarios:
  [model/observers/obs_vacc_rollout_scenarios.py](../model/observers/obs_vacc_rollout_scenarios.py).
- Vaccines delivered by type:
  [model/observers/obs_vacc_delivered.py](../model/observers/obs_vacc_delivered.py).
- Age-specific infections and infected counts:
  [model/observers/obs_prevalence_by_age.py](../model/observers/obs_prevalence_by_age.py).
- Disease events by age group:
  [model/observers/obs_disease_by_age.py](../model/observers/obs_disease_by_age.py).


# 3. Details <a name="details"></a>

## 3.1 Initialization

- `go_single()` in `varying_transmission_run.py` creates output path / file naming based on the year span
  and either loads an existing HDF5 disease file or creates a new one:
  [model/disease/varying_transmission_run.py](../model/disease/varying_transmission_run.py) (line 18).
- `DisSimulation.setup()` and `create_population()` build `DisPopulation`, set RNG / death rates,
  and generate the age-structured population:
  [model/disease/disease_simulation.py](../model/disease/disease_simulation.py) (lines 72, 89).
- Each new individual receives:
  - Demographics: `age`, `age_days`, `age_group` (= `age // 5`, capped at 15).
  - `quantile`: drawn once from `rng.rand()` — persistent.
  - `random`: drawn from `rng.rand()` — will be refreshed each tick.
  - `exp_random`: drawn from `rng.exponential(duration_of_infection)` — refreshed on each acquisition:
  [model/population/disease_population.py](../model/population/disease_population.py) (lines 124–141).
- When `read_population = True`, the initial population is instead restored from CSV triplets in
  `data/disease_pop_data/{pop_group}.csv`, `{pop_group}_endTimes.csv`, and `{pop_group}_strain_list.csv`:
  [model/population/disease_population.py](../model/population/disease_population.py) (lines 31, 35).
- Initial infection seeding uses an age-dependent probability (doubled for children ≤ 2 years);
  `random` and `exp_random` are refreshed for each age group during seeding:
  [model/disease/disease.py](../model/disease/disease.py) (lines 286, 318, 338, 347).

## 3.2 Input data

- Baseline model parameters are in `run_scenarios/base_params.py`
  (demography, transmission, vaccination, seeds, time scale, run horizon):
  [run_scenarios/base_params.py](../run_scenarios/base_params.py).
- Contact matrix is loaded at runtime from
  `data/population/all_contact_matrix_Australia_prem_2017.csv` as a NumPy array:
  [run_scenarios/varying_disease_model.py](../run_scenarios/varying_disease_model.py) (line 74).
- Strain list (`disease/strain_list.dat`) and vaccine configuration (`vaccine_configs/vaccine_list.dat`)
  are loaded in `Disease._load_disease_data()`:
  [model/disease/disease.py](../model/disease/disease.py) (lines 87, 163, 169).
- Age-specific infection durations, antibody distributions, and disease multipliers are loaded
  from `data/disease/` and `data/immunity/` sub-directories:
  [model/disease/disease.py](../model/disease/disease.py) (lines 130–160),
  [model/disease/antibody_levels.py](../model/disease/antibody_levels.py).
- Per-serotype-group transmission multipliers are read from `p['transmission_coefficient_multipliers']`
  in `VaryingTransmissionDisease._load_disease_data()`:
  [model/disease/varying_transmission_disease.py](../model/disease/varying_transmission_disease.py) (lines 38, 52).

## 3.3 Submodels

### 3.3.1 Demography submodel

- Each tick, ages advance by `period = 364 // t_per_year` days
  (`age += (age_days + period) // 364`, `age_days = (age_days + period) % 364`):
  [model/disease/disease_simulation.py](../model/disease/disease_simulation.py) (lines 240–248).
- Death is stochastic: each tick `random < death_rate[age]` determines survival;
  `random` is immediately refreshed after the check. There is no pre-assigned `days_at_death` column:
  [model/disease/disease_simulation.py](../model/disease/disease_simulation.py) (lines 252–256).
- `age_group` is recomputed after ageing (`age // 5`, capped at 15 for 80+):
  [model/disease/disease_simulation.py](../model/disease/disease_simulation.py) (lines 244–249).
- Birth and migration flows are computed each tick from per-tick rates, with fractional residue
  accumulation for births:
  [model/disease/disease_simulation.py](../model/disease/disease_simulation.py) (lines 271–285).
- New births and migrants are added via `introduce_births_and_migrations()`;
  births receive age 0, migrants are sampled from the migration age distribution:
  [model/disease/disease_utils.py](../model/disease/disease_utils.py).

### 3.3.2 Vaccination submodel

- Vaccine configuration is loaded from `vaccine_configs/vaccine_list.dat` (JSON) and expanded to
  year-aligned daily schedules during `Disease._load_disease_data()`:
  [model/disease/disease.py](../model/disease/disease.py) (lines 169, 175).
- At each tick `check_vaccines()` evaluates all active vaccine schedules, applies on-time and late
  coverage fractions, and updates the `vaccines` struct on eligible individuals:
  [model/disease/disease.py](../model/disease/disease.py) (line 534).
- The `vaccine_antibody_df` lookup table (vaccine × dose × serotype → lognormal antibody parameters)
  is built at initialization by `create_vaccine_antibody_df()` and joined at exposure and disease-outcome steps:
  [model/disease/antibody_levels.py](../model/disease/antibody_levels.py),
  [model/disease/disease.py](../model/disease/disease.py) (line 210).

### 3.3.3 Transmission and acquisition submodel

- Serotypes are reclassified into four groups at initialization:
  **pcv7** (7 original PCV7 serotypes), **pcv13** (6 PCV13 additions),
  **ppv23** (PPV23 serotypes not in PCV7/13), **nonppv23** (all others):
  [model/disease/varying_transmission_disease.py](../model/disease/varying_transmission_disease.py) (lines 54, 69, 86).
- `calc_foi()` (overridden in `VaryingTransmissionDisease`) groups the population by `age_group`,
  maps each carried strain to its vaccine group, and computes per-group infection fractions.
  `calc_age_group_fois()` then computes `transmission_coef × multiplier × Σ(inf_fraction × contact_row)` per group
  and sums across groups to get `prob_infection` per age group via $1-e^{-FOI}$:
  [model/disease/varying_transmission_disease.py](../model/disease/varying_transmission_disease.py) (lines 376, 411).
- In `check_exposure()`, all individuals are sorted by `random`, `no_of_strains`, `age_group`;
  each gets a strain assigned from the current `strain_distribution`. Infection fires if:
  `random <= prob_infection[age_group] * (1 - reduction * no_of_strains)` (with co-infection cap);
  `random` is refreshed after the comparison:
  [model/disease/disease.py](../model/disease/disease.py) (lines 950, 957, 960).
- For vaccinated individuals that pass the initial random check, `quantile` is used to place
  the individual on the antibody lognormal distribution; the resulting `waning_log_antibodies`
  gated by a logistic `prob_of_transmission` determines final acquisition:
  [model/disease/disease.py](../model/disease/disease.py) (lines 1027, 1041, 1052).
- On acquisition, `generate_duration_of_infection()` draws a fresh exponential from the
  age-specific mean duration and stores it in `exp_random`;
  clearance day = `day + period * (exp_random / period).round()` is appended to `endTimes`:
  [model/disease/disease.py](../model/disease/disease.py) (lines 1067, 1100, 1404).

### 3.3.4 Clearance / recovery submodel

- Within each call to `check_exposure()`, expired carriage events (where any `endTimes` entry has
  passed `day`) are detected, and the corresponding entries in `strain_list` and `endTimes` are
  removed; `no_of_strains` is updated:
  [model/disease/disease.py](../model/disease/disease.py) (lines 970–990).
- The final `update_ind_states()` call drops temporary columns (`will_infected`, `exposed_strains`);
  the persistent `exp_random` column retains the last drawn duration value for each individual:
  [model/disease/disease.py](../model/disease/disease.py) (line 1468).

### 3.3.5 Disease outcome submodel

- After exposure processing, `check_disease()` is called on **all currently infected individuals**
  (`P.I.filter(no_of_strains >= 1)`), not only on newly acquired infections:
  [model/disease/disease.py](../model/disease/disease.py) (line 1129).
- Individuals are joined with `vaccine_antibody_df` and `age_0_conversion`/`age1_conversion`
  tables to obtain `age_coef` (fine-grained age class for disease probability):
  [model/disease/disease.py](../model/disease/disease.py) (lines 1150, 1180).
- For vaccinated individuals, `quantile` is used to draw `log_antibodies` from the lognormal;
  waning is applied via `waning_ratio()`; disease probability uses a sigmoidal function
  of `waning_log_antibodies` scaled by `prob_dis_logantibody_scale[age_coef]`:
  [model/disease/disease.py](../model/disease/disease.py) (lines 1214, 1254).
- The disease draw `random <= prob_of_disease` determines whether disease occurs;
  a second `random` draw splits outcomes into `ipd` vs `cap` using `ipd_fraction_by_age_group`:
  [model/disease/disease.py](../model/disease/disease.py) (lines 1290, 1298).
- Disease events accumulate in `P.disease_pop` (reset at start of each year):
  [model/disease/disease.py](../model/disease/disease.py) (lines 1305, 1308).

### 3.3.6 External exposure submodel

- At intervals of `external_exposure_check_period` = `t_per_year // external_exposure_check_per_year`
  ticks, `external_exposure()` randomly samples strains and assigns them to susceptible
  unvaccinated individuals; each fires with probability `external_exposure_prob * (N / N_susceptible)`:
  [model/disease/disease.py](../model/disease/disease.py) (lines 1354, 1375).
- External infections go through the same `generate_duration_of_infection()` path,
  setting `exp_random` and appending `endTimes`:
  [model/disease/disease.py](../model/disease/disease.py) (lines 1388, 1404).

### 3.3.7 Population save / load submodel

- When `save_population = True`, the population is serialised to CSV triplets at the end of the run
  (note: `random`, `exp_random`, and `quantile` columns are included in the base file):
  [model/disease/disease_simulation.py](../model/disease/disease_simulation.py) (lines 163–183).
- When `read_population = True`, the saved state is restored from
  `data/disease_pop_data/{pop_group}.csv` and companion files;
  `random` and `exp_random` are **not** restored (they are refreshed on the first tick):
  [model/population/disease_population.py](../model/population/disease_population.py) (lines 31, 35).
- The `pop_group` and `pop_saving_address` parameters in `base_params.py` control which
  checkpoint files are read / written:
  [run_scenarios/base_params.py](../run_scenarios/base_params.py).


# 4. Code-Verified Single Timestep Trace <a name="single-timestep-trace"></a>

Given tick `t`: `day = t * 364 // t_per_year`.

1. **Demography update** (if enabled):
   - Each individual ages by `period = 364 // t_per_year` days (`age`, `age_days`, `age_group` recomputed).
   - `random < death_rate[age]` → individual removed; `random` refreshed for survivors.
   - Births and migrants introduced (each gets fresh `quantile`, `random`, `exp_random`).
   - Sources: [model/disease/disease_simulation.py](../model/disease/disease_simulation.py)
     (lines 133, 137, 226, 240–256).

2. **Vaccination update** (`check_vaccines`):
   - Active schedules evaluated; on-time and late doses assigned based on rollout year and coverage fractions.
   - `vaccines` struct updated in place (`no_of_doses`, `vaccine_type`, `final_vaccine_time`).
   - Sources: [model/disease/disease.py](../model/disease/disease.py) (lines 515, 534).

3. **FOI computation** (`VaryingTransmissionDisease.calc_foi`):
   - Population grouped by `age_group`; strains mapped to vaccine groups (pcv7/pcv13/ppv23/nonppv23).
   - Per-group infection fractions and `calc_age_group_fois()` produce `prob_infection[age_group]`
     via $1-e^{-\text{transmission\_coef} \times \text{multiplier} \times \text{FOI}}$.
   - Current `strain_distribution` (with optional noise) is computed for strain assignment.
   - Sources: [model/disease/varying_transmission_disease.py](../model/disease/varying_transmission_disease.py)
     (lines 376, 411, 448).

4. **Exposure and clearance** (`check_exposure`) — if any individuals are infected:
   - Individuals sorted by `random`, `no_of_strains`, `age_group`; each assigned an `exposed_strains` sample.
   - Exposure fires: `random <= prob_infection[age_group] * (1 - reduction * no_of_strains)`; `random` refreshed.
   - For vaccinated individuals that pass: `quantile` → `log_antibodies` via lognormal ppf; waning applied;
     logistic `prob_of_transmission` used for a second check.
   - For those that will acquire: `generate_duration_of_infection()` draws fresh `exp_random`;
     `endTimes += day + period * (exp_random / period).round()`; `strain_list` and `no_of_strains` updated.
   - Expired carriage (any `endTimes <= day`) removed from `strain_list`, `endTimes`, `no_of_strains`.
   - **Disease outcome** (`check_disease`) called on all currently infected (`no_of_strains >= 1`):
     `quantile` → `log_antibodies` → `waning_log_antibodies`; `random <= prob_of_disease` → disease flag;
     second `random` draw splits into `ipd` / `cap`; events added to `P.disease_pop`.
   - Sources: [model/disease/disease.py](../model/disease/disease.py)
     (lines 927, 950, 957, 970, 1027, 1041, 1067, 1100, 1129, 1141, 1214, 1290, 1298, 1305).

5. **External introductions** (`external_exposure`) — if `day % external_exposure_check_period == 0`:
   - Random strain sample assigned to susceptible unvaccinated individuals;
     fires with probability `external_exposure_prob * (N / N_susceptible)`.
   - `generate_duration_of_infection()` refreshes `exp_random`; `endTimes` updated.
   - Sources: [model/disease/disease.py](../model/disease/disease.py) (lines 1354, 1388, 1404).

6. **Individual state finalization** (`update_ind_states`):
   - Temporary columns `will_infected` and `exposed_strains` dropped from `P.I`.
   - Source: [model/disease/disease.py](../model/disease/disease.py) (line 1468).

7. **Observer writes**: all attached observers record their summaries for this tick.
   - Source: [model/disease/disease.py](../model/disease/disease.py) (line 1478),
     [run_scenarios/varying_disease_model.py](../run_scenarios/varying_disease_model.py) (lines 50, 62).

This sequence repeats for all ticks in `[years[0] * t_per_year, years[1] * t_per_year]`:
[run_scenarios/base_params.py](../run_scenarios/base_params.py),
[model/disease/disease_simulation.py](../model/disease/disease_simulation.py) (lines 115–147).
